# Nemotron 3 Nano Omni 30B

Calls `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` on NVIDIA NIM. Omni reads images, video, audio, and text, and it returns a written answer. This notebook asks it to describe NVIDIA's sample image.

## Runtime

These models are hosted NVIDIA NIMs. Colab sends the prompt to `https://integrate.api.nvidia.com` and NVIDIA's GPUs do the inference. A Colab GPU cannot load the 30B or 550B models, and requesting one can fail when Colab GPUs are busy. Leave the runtime on the default CPU. Colab Pro is useful because the session can stay connected while a long reply streams back.

## API key

The notebook reads a Colab secret named `NVIDIA_API_KEY`.

1. In Colab, click the key icon in the left sidebar.
2. Add a secret named `NVIDIA_API_KEY`.
3. Paste a key from [build.nvidia.com](https://build.nvidia.com/).
4. Enable notebook access for that secret.
5. Choose **Runtime → Run all**.

If the secret is not set, the next cell asks for the key with a hidden prompt. The key stays in the runtime only. Do not paste it into a cell you save or commit.


In [ ]:
import platform
import shutil
import subprocess

print("Python", platform.python_version())
print("Inference runs on NVIDIA NIM GPUs. This VM only needs internet access.")
if shutil.which("nvidia-smi"):
    listed = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    print((listed.stdout or listed.stderr).strip() or "nvidia-smi returned no GPU list.")
else:
    print("No local NVIDIA GPU is attached. That is the expected Colab CPU runtime.")


In [ ]:
%pip install -q "openai>=1.54.0,<3" "requests>=2.31.0"


## Describe an image

The image is NVIDIA's public sample. Change `QUESTION` or `IMAGE_URL` to point at another public `https` image. Temperature `0.6` and `top_p` `0.95` match the published sample for this model.

In [ ]:
import os
import time
from getpass import getpass

import requests
from IPython.display import Image, display


def load_nvidia_api_key() -> str:
    key = os.environ.get("NVIDIA_API_KEY", "").strip()
    if key:
        return key
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    if userdata is not None:
        try:
            secret = userdata.get("NVIDIA_API_KEY")
        except Exception:
            secret = None
        if secret and str(secret).strip():
            key = str(secret).strip()
            os.environ["NVIDIA_API_KEY"] = key
            return key
    key = getpass("Enter your NVIDIA API key: ").strip()
    if not key:
        raise RuntimeError(
            "No NVIDIA API key was provided. Add a Colab secret named "
            "NVIDIA_API_KEY, allow notebook access, and run this cell again."
        )
    os.environ["NVIDIA_API_KEY"] = key
    return key


def as_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(str(item.get("text") or ""))
        return "".join(parts)
    return str(value)


NVIDIA_API_KEY = load_nvidia_api_key()
print(f"NVIDIA API key is ready ({len(NVIDIA_API_KEY)} characters). The key is not printed.")

# Colab-safe defaults. This model accepts max_tokens up to 65536.
# A long budget can sit until the gateway times out, so the demo stays modest.
QUESTION = "What is in this image?"
IMAGE_URL = "https://assets.ngc.nvidia.com/products/api-catalog/phi-3-5-vision/example1b.jpg"
MAX_TOKENS = 2048
REASONING_BUDGET = 1024
MODEL = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning"

display(Image(url=IMAGE_URL, width=640))
print(f"Model: {MODEL}", flush=True)
print("Waiting for the image reply...", flush=True)

payload = {
    "model": MODEL,
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": QUESTION},
                {"type": "image_url", "image_url": {"url": IMAGE_URL}},
            ],
        }
    ],
    "max_tokens": MAX_TOKENS,
    "reasoning_budget": REASONING_BUDGET,
    "stream": False,
    "temperature": 0.6,
    "top_p": 0.95,
}
headers = {
    "Authorization": f"Bearer {NVIDIA_API_KEY}",
    "Accept": "application/json",
}

response = None
for attempt in range(2):
    try:
        response = requests.post(
            "https://integrate.api.nvidia.com/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=300,
        )
    except requests.RequestException as exc:
        if attempt == 1:
            raise RuntimeError(f"Could not reach NVIDIA NIM: {exc}") from None
        print(f"Temporary network error ({type(exc).__name__}). Retrying once...", flush=True)
        time.sleep(3)
        continue
    if response.status_code in (401, 403):
        raise RuntimeError(
            "NVIDIA rejected the API key. Create a key at https://build.nvidia.com/ "
            "and save it as the Colab secret NVIDIA_API_KEY."
        )
    if response.status_code in (429, 500, 502, 503, 504) and attempt == 0:
        print(f"Temporary NIM error ({response.status_code}). Retrying once...", flush=True)
        time.sleep(3)
        continue
    break

if response is None or not response.ok:
    detail = "" if response is None else response.text[:500]
    status = "no response" if response is None else response.status_code
    raise RuntimeError(f"NVIDIA NIM request failed ({status}): {detail}")

data = response.json()
choices = data.get("choices") or []
if not choices:
    raise RuntimeError(f"NVIDIA NIM returned no choices: {str(data)[:500]}")
message = choices[0].get("message") or {}
reasoning = as_text(message.get("reasoning_content")).strip()
answer = as_text(message.get("content")).strip()
if reasoning:
    print("\n--- reasoning ---\n")
    print(reasoning)
print("\n--- answer ---\n")
print(answer)
if not answer:
    raise RuntimeError(
        "NVIDIA NIM returned no answer text. Increase MAX_TOKENS and run the cell again."
    )
